In [38]:
# ------------------------------
# STEP 1: Setup imports and load data from 02-feature-engineering.py
# ------------------------------
# Core libraries
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
import pickle
import os

# Machine Learning libraries
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("=" * 60)
print("STEP 1: LOADING DATA FROM 02-FEATURE-ENGINEERING.PY")
print("=" * 60)
# ------------------------------
# 1.1 Set up file paths
# ------------------------------
_base_dir = Path.cwd()
START_YEAR = 2000
# Input data path (from 02-feature-engineering.py)
data_path = _base_dir / f"signals_with_returns_and_tickers_{START_YEAR}.parquet"

# Create output directories for results
output_dir = _base_dir / "ml_ff_result"
print(_base_dir)
print(output_dir)

cv_dir = output_dir / "CV"  # Cross-validation results
pred_dir = output_dir / "Pred"  # Predictions

# Create directories if they don't exist
output_dir.mkdir(exist_ok=True)
cv_dir.mkdir(exist_ok=True)
pred_dir.mkdir(exist_ok=True)

print(f"Data path: {data_path}")
print(f"Output directory: {output_dir}")
print(f"CV directory: {cv_dir}")
print(f"Prediction directory: {pred_dir}")

# ------------------------------
# 1.2 Load data
# ------------------------------
print(f"\nLoading data from: {data_path}")
df = pd.read_parquet(data_path, engine="fastparquet")

print(f"Data loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Rows: {len(df):,}")

# ------------------------------
# 1.3 Ensure date columns are datetime
# ------------------------------
for c in ("datadate", "form_date"):
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

print(f"\nDate range: {df['form_date'].min()} to {df['form_date'].max()}")

# ------------------------------
# 1.4 Calculate expected_return if not exists
# ------------------------------
if 'expected_return' not in df.columns:
    print("\nCalculating expected_return = ret - rf...")
    df['expected_return'] = df['ret'] - df['rf']
else:
    print("\nExpected return already exists")

print(f"Expected return - Mean: {df['expected_return'].mean():.4f}, Std: {df['expected_return'].std():.4f}")

# ------------------------------
# 1.5 Identify base columns vs feature columns
# ------------------------------
print("\nIdentifying base columns and feature columns...")

# Base/metadata columns (not used as features)
base_columns = [
    'gvkey', 'datadate', 'fyear', 'year', 'permno', 'form_date', 'form_year',
    'crsp_mktcap_6', 'ret', 'rf', 'nmonth', 'i', 'mindex',
    'mindex_form', 'ticker', 'expected_return', 'counter'
]

# Filter to only base columns that exist in df
base_columns = [col for col in base_columns if col in df.columns]

# All other columns are potential features
all_columns = df.columns.tolist()
feature_columns = [col for col in all_columns if col not in base_columns]

print(f"Base/metadata columns: {len(base_columns)}")
print(f"Feature columns: {len(feature_columns)}")

# Show sample of base columns
print(f"\nBase columns: {base_columns}")

# Show first 10 feature columns
print(f"\nFirst 10 feature columns: {feature_columns[:10]}")

print("=" * 60)
print("STEP 1 COMPLETE: Data loaded successfully!")
print("=" * 60)

STEP 1: LOADING DATA FROM 02-FEATURE-ENGINEERING.PY
/Users/natasha/Documents/GitHub/qf600_asset_pricing/yz_data
/Users/natasha/Documents/GitHub/qf600_asset_pricing/yz_data/ml_ff_result
Data path: /Users/natasha/Documents/GitHub/qf600_asset_pricing/yz_data/signals_with_returns_and_tickers_2000.parquet
Output directory: /Users/natasha/Documents/GitHub/qf600_asset_pricing/yz_data/ml_ff_result
CV directory: /Users/natasha/Documents/GitHub/qf600_asset_pricing/yz_data/ml_ff_result/CV
Prediction directory: /Users/natasha/Documents/GitHub/qf600_asset_pricing/yz_data/ml_ff_result/Pred

Loading data from: /Users/natasha/Documents/GitHub/qf600_asset_pricing/yz_data/signals_with_returns_and_tickers_2000.parquet
Data loaded successfully!
Dataset shape: (66459, 20189)
Columns: 20189
Rows: 66,459

Date range: 2002-06-28 00:00:00 to 2024-06-28 00:00:00

Calculating expected_return = ret - rf...
Expected return - Mean: 0.1018, Std: 0.6945

Identifying base columns and feature columns...
Base/metadata c

In [54]:
# ------------------------------
# 1.6 Simplifying our features by only using the FF3 data as our features
# ------------------------------
# 3 independent varibales taken to the model: Market-Rf (Market Excess Return), SMB (Size factor), HML (Value factor) ONLY

col = ['ticker','form_date', 'form_year','datadate','ret','rf','expected_return'] #expected return is actually referring to excess return
price_df = df[col]
print("Stock Characteristics (Annual)")
print(price_df.head())
print("-"*60)

print("Loading the monthly data data (Monthly")
data_path2 = _base_dir/"fama_french"/f"ff3_monthly.csv"
ff3_df = pd.read_csv(data_path2)
ff3_df.rename(columns={'Unnamed: 0':'form_date'},inplace=True)
print(ff3_df.head())
print("-"*60)

print("Calibrating the FF3 to Annual Based")
ff3_df['form_date'] = pd.to_datetime(ff3_df['form_date'], format='%Y%m', errors='coerce')
ff3_df['year'] = ff3_df['form_date'].dt.year
ff3_df['month'] = ff3_df['form_date'].dt.month
ff3_df = ff3_df.sort_values('form_date')

# Create fiscal year (July to June)
# If month >= 7 (July onwards), it's the start of the fiscal year
# Fiscal year labeled by the ending year (e.g., July 2003 - June 2004 = FY 2004)
ff3_df['fiscal_year'] = ff3_df.apply(
    lambda x: x['year'] + 1 if x['month'] >= 7 else x['year'], 
    axis=1
)

print("\nAggregating monthly factors to fiscal years (July-June)...")
print("Example: July 2003 - June 2004 = Fiscal Year 2004")

# Aggregate by fiscal year
ff3_annual = ff3_df.groupby('fiscal_year').agg({
    'Mkt-RF': lambda x: (1 + x/100).prod() - 1 if (x/100).abs().mean() > 0.1 else x.sum(),
    'SMB': lambda x: (1 + x/100).prod() - 1 if (x/100).abs().mean() > 0.1 else x.sum(),
    'HML': lambda x: (1 + x/100).prod() - 1 if (x/100).abs().mean() > 0.1 else x.sum(),
    'form_date': 'count'
}).reset_index()

ff3_annual = ff3_annual.rename(columns={'form_date': 'n_months', 'fiscal_year': 'year'})

# Filter years with full data (should have 12 months)
MIN_MONTHS = 12
ff3_annual = ff3_annual[ff3_annual['n_months'] >= MIN_MONTHS]

print(f"Annual FF3 factors: {len(ff3_annual)} years")
print(f"Fiscal year range: FY{ff3_annual['year'].min()} to FY{ff3_annual['year'].max()}")
print("\nAnnual FF3 factors (Fiscal Years):")
print(ff3_annual.head())
print("\nNote: Each fiscal year runs from July to June")
print("Example: FY2004 = July 2003 through June 2004")
print("-"*20)

#combining ff3 data based on the stock information

ff3f = pd.merge(price_df, ff3_annual, left_on='form_year',right_on='year', how='left')
ff3f = ff3f.drop(['form_date','datadate','ret','rf','year','n_months'],axis=1)
ff3f.rename(columns={'expected_return':'excess_return'},inplace=True)
print("final ff3f data for final use")
print(ff3f.head())
print("-"*60)

Stock Characteristics (Annual)
  ticker  form_date  form_year   datadate       ret        rf  expected_return
0    AIR 2003-06-30     2003.0 2002-05-31  0.607647  0.008836         0.598811
1    AIR 2004-06-30     2004.0 2003-05-31  0.384137  0.019674         0.364463
2    AIR 2005-06-30     2005.0 2004-05-31  0.415024  0.040532         0.374491
3    AIR 2006-06-30     2006.0 2005-05-31  0.484927  0.050743         0.434184
4    AIR 2007-06-29     2007.0 2006-05-31 -0.590124  0.031542        -0.621665
------------------------------------------------------------
Loading the monthly data data (Monthly
   form_date  Mkt-RF   SMB   HML    RF
0     192607    2.89 -2.55 -2.39  0.22
1     192608    2.64 -1.14  3.81  0.25
2     192609    0.38 -1.36  0.05  0.23
3     192610   -3.27 -0.14  0.82  0.32
4     192611    2.54 -0.11 -0.61  0.31
------------------------------------------------------------
Calibrating the FF3 to Annual Based

Aggregating monthly factors to fiscal years (July-June)...
Exam

In [58]:
# ------------------------------
# STEP 2: Create a beta estimation for all factors
# ------------------------------
# Function to estimate FF3 betas for each stock
def calculate_beta_up_to_year(ticker_data, target_year, min_years=3):
    """
    Calculate beta using only data UP TO (and including) target_year
    
    Parameters:
    -----------
    ticker_data : DataFrame
        All data for one ticker
    target_year : float
        Calculate beta using data up to this year (inclusive)
    min_years : int
        Minimum number of years needed for calculation
    
    Returns:
    --------
    dict : Betas and statistics, or None if insufficient data
    """
    # Filter data up to target year (inclusive)
    historical_data = ticker_data[ticker_data['form_year'] <= target_year].copy()
    
    if len(historical_data) < min_years:
        return None
    
    # Prepare regression data
    X = historical_data[['Mkt-RF', 'SMB', 'HML']].values / 100  # Convert % to decimal
    y = historical_data['excess_return'].values
    
    # Remove NaN
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X = X[mask]
    y = y[mask]
    
    if len(y) < min_years:
        return None
    
    # Run regression
    model = LinearRegression()
    model.fit(X, y)
    
    # Calculate R-squared
    y_pred = model.predict(X)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    return {
        'alpha': model.intercept_,
        'beta_mkt': model.coef_[0],
        'beta_smb': model.coef_[1],
        'beta_hml': model.coef_[2],
        'r_squared': r_squared,
        'n_years': len(y),
        'years_used': f"{historical_data['form_year'].min():.0f}-{historical_data['form_year'].max():.0f}"
    }

# ============================================================================
# CALCULATE ROLLING BETAS FOR ALL TICKERS AND YEARS
# ============================================================================
print("\n[STEP 2] Calculating rolling betas for each ticker-year combination...")

all_results = []
tickers = ff3f['ticker'].unique()
total_combinations = 0

for i, ticker in enumerate(tickers):
    if (i + 1) % 100 == 0:
        print(f"  Processing ticker {i+1}/{len(tickers)}...")
    
    # Get all data for this ticker
    ticker_data = ff3f[ff3f['ticker'] == ticker].sort_values('form_year')
    
    # Get all years for this ticker
    years = ticker_data['form_year'].unique()
    
    # Calculate beta for each year using only data up to that year
    for year in years:
        betas = calculate_beta_up_to_year(ticker_data, year, min_years=3)
        
        if betas is not None:
            # Get the actual factor values for this year
            year_factors = ticker_data[ticker_data['form_year'] == year].iloc[0]
            
            result = {
                'ticker': ticker,
                'form_year': year,
                'alpha': betas['alpha'],
                'beta_mkt': betas['beta_mkt'],
                'beta_smb': betas['beta_smb'],
                'beta_hml': betas['beta_hml'],
                'r_squared': betas['r_squared'],
                'n_years_used': betas['n_years'],
                'years_range': betas['years_used'],
                # Store factor values for this year
                'mkt_rf': year_factors['Mkt-RF'],
                'smb': year_factors['SMB'],
                'hml': year_factors['HML'],
                'actual_excess_return': year_factors['excess_return']
            }
            
            # Calculate expected return using the betas estimated up to this year
            result['expected_return'] = (
                betas['alpha'] +
                betas['beta_mkt'] * (year_factors['Mkt-RF'] / 100) +
                betas['beta_smb'] * (year_factors['SMB'] / 100) +
                betas['beta_hml'] * (year_factors['HML'] / 100)
            )
            
            # Calculate prediction error (alpha)
            result['prediction_error'] = result['actual_excess_return'] - result['expected_return']
            
            all_results.append(result)
            total_combinations += 1

results_df = pd.DataFrame(all_results)

print(f"\n✓ Calculated rolling betas for {total_combinations} ticker-year combinations")
print(f"  Unique tickers: {results_df['ticker'].nunique()}")
print(f"  Years covered: {results_df['form_year'].min():.0f} to {results_df['form_year'].max():.0f}")
print(results_df.head())
print("-"*60)


[STEP 2] Calculating rolling betas for each ticker-year combination...
  Processing ticker 100/7401...
  Processing ticker 200/7401...
  Processing ticker 300/7401...
  Processing ticker 400/7401...
  Processing ticker 500/7401...
  Processing ticker 600/7401...
  Processing ticker 700/7401...
  Processing ticker 800/7401...
  Processing ticker 900/7401...
  Processing ticker 1000/7401...
  Processing ticker 1100/7401...
  Processing ticker 1200/7401...
  Processing ticker 1300/7401...
  Processing ticker 1400/7401...
  Processing ticker 1500/7401...
  Processing ticker 1600/7401...
  Processing ticker 1700/7401...
  Processing ticker 1800/7401...
  Processing ticker 1900/7401...
  Processing ticker 2000/7401...
  Processing ticker 2100/7401...
  Processing ticker 2200/7401...
  Processing ticker 2300/7401...
  Processing ticker 2400/7401...
  Processing ticker 2500/7401...
  Processing ticker 2600/7401...
  Processing ticker 2700/7401...
  Processing ticker 2800/7401...
  Processing 

In [59]:

# ============================================================================
# Step 2.5: DISPLAY SAMPLE RESULTS
# ============================================================================
print("\n" + "="*80)
print("SAMPLE RESULTS - TICKER: AIR")
print("="*80)

sample_ticker = results_df[results_df['ticker'] == 'AIR'].sort_values('form_year')
print("\nShowing how beta evolves as more data becomes available:")
print(f"{'Year':<8} {'Years Used':<15} {'Beta Mkt':<10} {'Beta SMB':<10} {'Beta HML':<10} {'R²':<8} {'Expected':<12} {'Actual':<12}")
print("-"*100)
for _, row in sample_ticker.iterrows():
    print(f"{row['form_year']:<8.0f} {row['years_range']:<15} "
          f"{row['beta_mkt']:>8.3f} {row['beta_smb']:>8.3f} {row['beta_hml']:>8.3f} "
          f"{row['r_squared']:>6.3f} {row['expected_return']*100:>10.2f}% {row['actual_excess_return']*100:>10.2f}%")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print("\nBeta Statistics (Overall):")
print(results_df[['beta_mkt', 'beta_smb', 'beta_hml', 'r_squared']].describe())

print("\nPrediction Accuracy:")
print(f"Mean prediction error: {results_df['prediction_error'].mean()*100:.4f}%")
print(f"Mean absolute error: {results_df['prediction_error'].abs().mean()*100:.4f}%")
print(f"RMSE: {np.sqrt((results_df['prediction_error']**2).mean())*100:.4f}%")

# Correlation between expected and actual
correlation = results_df['expected_return'].corr(results_df['actual_excess_return'])
print(f"Correlation (Expected vs Actual): {correlation:.4f}")

# ============================================================================
# ANALYZE BY NUMBER OF YEARS USED
# ============================================================================
print("\n" + "="*80)
print("BETA STABILITY BY NUMBER OF YEARS USED")
print("="*80)

years_analysis = results_df.groupby('n_years_used').agg({
    'r_squared': 'mean',
    'prediction_error': lambda x: x.abs().mean(),
    'ticker': 'count'
}).rename(columns={'ticker': 'count'})

print("\n", years_analysis)

# ============================================================================
# SAVE RESULTS
# ============================================================================
print("\n[STEP 3] Saving results...")

# Save all rolling betas
output_path = output_dir / 'rolling_betas_by_year.csv'
results_df.to_csv(output_path, index=False)
print(f"✓ Rolling betas: {output_path}")

# Save latest year betas only (for portfolio construction)
latest_year = results_df['form_year'].max()
latest_betas = results_df[results_df['form_year'] == latest_year].copy()
latest_path = output_dir / f'betas_for_year_{latest_year:.0f}.csv'
latest_betas.to_csv(latest_path, index=False)
print(f"✓ Latest year ({latest_year:.0f}) betas: {latest_path}")
print(latest_betas.head())


SAMPLE RESULTS - TICKER: AIR

Showing how beta evolves as more data becomes available:
Year     Years Used      Beta Mkt   Beta SMB   Beta HML   R²       Expected     Actual      
----------------------------------------------------------------------------------------------------
2005     2003-2005         -0.589   -0.045   -1.181  1.000      37.45%      37.45%
2006     2003-2006         -0.392   -0.267   -1.271  1.000      43.42%      43.42%
2007     2003-2007         -6.514    6.978    1.465  0.971     -59.81%     -62.17%
2008     2003-2008         -4.714    6.672    2.501  0.867      30.91%      18.03%
2009     2003-2009         -0.864    1.573    1.773  0.209      38.84%       4.22%
2010     2003-2010         -0.754    1.879    1.969  0.269      41.93%      62.20%
2011     2003-2011         -1.311    1.499    2.935  0.464     -38.50%     -49.52%
2012     2003-2012         -1.109   -0.182    2.792  0.305      11.75%      66.03%
2013     2003-2013         -1.156    0.014    2.588  0

In [68]:
ff3f.head()

,ticker,form_year,excess_return,Mkt-RF,SMB,HML
0,AIR,2003.0,0.598811,2.30,3.33,-4.25
1,AIR,2004.0,0.364463,18.59,13.18,7.09
2,AIR,2005.0,0.374491,5.81,-0.58,13.14
3,AIR,2006.0,0.434184,5.55,4.19,7.52
4,AIR,2007.0,-0.621665,13.76,-2.93,-1.54


In [71]:
results_df['prediction_error_pct'] = (results_df['prediction_error']*100).round(2)
results_df.head()

,ticker,form_year,alpha,beta_mkt,beta_smb,beta_hml,r_squared,n_years_used,years_range,mkt_rf,smb,hml,actual_excess_return,expected_return,prediction_error,prediction_error_pct
0,AIR,2005.0,0.563662,-0.589128,-0.045037,-1.181152,1.000000,3,2003-2005,5.81,-0.58,13.14,0.374491,0.374491,-5.551115e-17,-0.00
1,AIR,2006.0,0.562723,-0.392327,-0.267254,-1.270839,1.000000,4,2003-2006,5.55,4.19,7.52,0.434184,0.434184,0.000000e+00,0.00
2,AIR,2007.0,0.525218,-6.513883,6.977990,1.464999,0.971046,5,2003-2007,13.76,-2.93,-1.54,-0.621665,-0.598108,-2.355739e-02,-2.36
3,AIR,2008.0,0.298905,-4.714410,6.672147,2.500868,0.867342,6,2003-2008,-15.75,-5.89,-13.57,0.180339,0.309067,-1.287277e-01,-12.87
4,AIR,2009.0,0.153215,-0.863579,1.572714,1.772982,0.208787,7,2003-2009,-25.16,8.78,-6.78,0.042188,0.388368,-3.461801e-01,-34.62


In [88]:
# ------------------------------
# STEP 3: Create portfolio based on top 50 and bottom 50
# ------------------------------

# Number of stocks to sell (short)
INITIAL_CAPITAL = 1_000_000 
print(f"\n📊 Creating long/short portfolios...")
print(f"💰 Total capital: ${INITIAL_CAPITAL:,.0f}")
print(f"   Capital allocation: Flexible based on number of positions")

# Create portfolio dataset
predictions_df = results_df.copy()
# ========================================
# VERSION 1: FIXED TOP 50 / BOTTOM 50
# ========================================
print(f"\n📈 Building portfolios - VERSION 1: Fixed Top 100 / Bottom 100")
print("-" * 60)
# ============================================================================
# BUILD LONG-SHORT PORTFOLIO
# ============================================================================
print("\n[STEP 2] Building long-short portfolio strategy...")

# Get all unique years (sorted)
all_years = sorted(predictions_df['form_year'].unique())
TOP_N = 100  # Number of stocks to buy (long)
BOTTOM_N = 100
# We need at least 2 years (formation year + holding year)
if len(all_years) < 2:
    print("ERROR: Need at least 2 years of data")
    exit(1)
    \

portfolio_results = []

print(f"\nBacktesting from {all_years[0]:.0f} to {all_years[-1]:.0f}")
print("Strategy: Sort by predicted return in Year T, hold in Year T+1")

for i in range(len(all_years) - 1):
    formation_year = all_years[i]
    holding_year = all_years[i + 1]
    
    # Get predictions from formation year
    formation_data = predictions_df[predictions_df['form_year'] == formation_year].copy()
    
    # Sort by expected return
    formation_data = formation_data.sort_values('expected_return', ascending=False)
    
    # Check if we have enough stocks
    if len(formation_data) < (TOP_N + BOTTOM_N):
        print(f"  Skipping {formation_year:.0f}: Not enough stocks")
        continue
    
    # Select top N for long, bottom N for short
    long_tickers = formation_data.head(TOP_N)['ticker'].tolist()
    short_tickers = formation_data.tail(BOTTOM_N)['ticker'].tolist()

    
    
    # Get actual returns in the holding year
    holding_data = predictions_df[predictions_df['form_year'] == holding_year].copy()
    
    # Get returns for long portfolio
    long_returns = holding_data[holding_data['ticker'].isin(long_tickers)].copy()
    
    # Get returns for short portfolio
    short_returns = holding_data[holding_data['ticker'].isin(short_tickers)].copy()
    print(long_returns)
    # Check if we have returns for all stocks
    n_long = len(long_returns)
    n_short = len(short_returns)
    
    if n_long == 0 or n_short == 0:
        print(f"  Skipping {formation_year:.0f}→{holding_year:.0f}: Missing return data")
        continue
    
    # ========================================================================
    # EQUAL WEIGHT ALLOCATION
    # ========================================================================
    
    # Allocate capital
    long_capital = INITIAL_CAPITAL / 2  # Half for long
    short_capital = INITIAL_CAPITAL / 2  # Half for short
    
    # Position size per stock (equal weight)
    position_size_long = long_capital / n_long
    position_size_short = short_capital / n_short
    
    # Calculate portfolio returns
    # Long: Buy stocks, profit when they go up
    long_return = (long_returns['actual_excess_return']+long_returns['mkt_rf']).mean()
    
    # Short: Sell stocks, profit when they go down
    short_return = -(short_returns['actual_excess_return']+short_returns['mkt_rf']).mean()  # Negative because we're short
    
    # Portfolio return (equal weight between long and short)
    portfolio_return = (long_return + short_return) / 2
    
    # Long-Short spread (long return minus actual short return, not inverted)
    spread = long_return-short_return
    
    # ========================================================================
    # DOLLAR-BASED CALCULATIONS
    # ========================================================================
    
    # Dollar P&L for long portfolio
    dollar_pnl_long = long_capital * long_return
    
    # Dollar P&L for short portfolio
    dollar_pnl_short = short_capital * short_return
    
    # Total dollar P&L
    total_dollar_pnl = dollar_pnl_long + dollar_pnl_short
    
    # Store results
    portfolio_results.append({
        'formation_year': formation_year,
        'holding_year': holding_year,
        'long_return': long_return,
        'short_return': short_return,
        'spread': spread,
        'portfolio_return': portfolio_return,
        'n_long': n_long,
        'n_short': n_short,
        # Dollar-based metrics
        'long_capital': long_capital,
        'short_capital': short_capital,
        'position_size_long': position_size_long,
        'position_size_short': position_size_short,
        'dollar_pnl_long': dollar_pnl_long,
        'dollar_pnl_short': dollar_pnl_short,
        'total_dollar_pnl': total_dollar_pnl
    })
    
    print(f"  {formation_year:.0f}→{holding_year:.0f}: Long={n_long} stocks ({long_return*100:>6.2f}%), "
          f"Short={n_short} stocks ({short_return*100:>6.2f}%), "
          f"Portfolio={portfolio_return*100:>6.2f}%, "
          f"P&L=${total_dollar_pnl:>10,.0f}")

# Convert to DataFrame
portfolio_df = pd.DataFrame(portfolio_results)

if len(portfolio_df) == 0:
    print("\nERROR: No portfolio results generated")
    exit(1)

# ============================================================================
# CALCULATE CUMULATIVE PERFORMANCE
# ============================================================================
print("\n[STEP 3] Calculating cumulative performance...")

# Calculate cumulative returns
portfolio_df['cumulative_long'] = (1 + portfolio_df['long_return']).cumprod() - 1
portfolio_df['cumulative_short'] = (1 + portfolio_df['short_return']).cumprod() - 1
portfolio_df['cumulative_portfolio'] = (1 + portfolio_df['portfolio_return']).cumprod() - 1

# Calculate cumulative dollar value
portfolio_df['total_capital'] = INITIAL_CAPITAL * (1 + portfolio_df['cumulative_portfolio'])
portfolio_df['cumulative_pnl'] = portfolio_df['total_dollar_pnl'].cumsum()

# ============================================================================
# PERFORMANCE METRICS
# ============================================================================
print("\n" + "="*80)
print("PORTFOLIO PERFORMANCE SUMMARY")
print("="*80)

# Basic statistics
total_years = len(portfolio_df)
avg_long_return = portfolio_df['long_return'].mean() * 100
avg_short_return = portfolio_df['short_return'].mean() * 100
avg_portfolio_return = portfolio_df['portfolio_return'].mean() * 100
avg_spread = portfolio_df['spread'].mean() * 100

print(f"\nBasic Statistics ({total_years} years):")
print(f"  Average long return: {avg_long_return:.2f}%")
print(f"  Average short return: {avg_short_return:.2f}%")
print(f"  Average portfolio return: {avg_portfolio_return:.2f}%")
print(f"  Average long-short spread: {avg_spread:.2f}%")

# Risk metrics
volatility_long = portfolio_df['long_return'].std() * 100
volatility_short = portfolio_df['short_return'].std() * 100
volatility_portfolio = portfolio_df['portfolio_return'].std() * 100

print(f"\nVolatility (Standard Deviation):")
print(f"  Long portfolio: {volatility_long:.2f}%")
print(f"  Short portfolio: {volatility_short:.2f}%")
print(f"  Combined portfolio: {volatility_portfolio:.2f}%")

# Sharpe ratio (assuming 0% risk-free rate for simplicity)
sharpe_long = avg_long_return / volatility_long if volatility_long > 0 else 0
sharpe_short = avg_short_return / volatility_short if volatility_short > 0 else 0
sharpe_portfolio = avg_portfolio_return / volatility_portfolio if volatility_portfolio > 0 else 0

print(f"\nSharpe Ratio (annualized, RF=0%):")
print(f"  Long portfolio: {sharpe_long:.3f}")
print(f"  Short portfolio: {sharpe_short:.3f}")
print(f"  Combined portfolio: {sharpe_portfolio:.3f}")

# Win rate
win_rate_long = (portfolio_df['long_return'] > 0).sum() / total_years * 100
win_rate_short = (portfolio_df['short_return'] > 0).sum() / total_years * 100
win_rate_portfolio = (portfolio_df['portfolio_return'] > 0).sum() / total_years * 100

print(f"\nWin Rate:")
print(f"  Long portfolio: {win_rate_long:.1f}%")
print(f"  Short portfolio: {win_rate_short:.1f}%")
print(f"  Combined portfolio: {win_rate_portfolio:.1f}%")

# Cumulative returns
total_return_long = portfolio_df['cumulative_long'].iloc[-1] * 100
total_return_short = portfolio_df['cumulative_short'].iloc[-1] * 100
total_return_portfolio = portfolio_df['cumulative_portfolio'].iloc[-1] * 100

print(f"\nCumulative Returns:")
print(f"  Long portfolio: {total_return_long:.2f}%")
print(f"  Short portfolio: {total_return_short:.2f}%")
print(f"  Combined portfolio: {total_return_portfolio:.2f}%")

# Dollar performance
final_capital = portfolio_df['total_capital'].iloc[-1]
total_pnl = portfolio_df['cumulative_pnl'].iloc[-1]

print(f"\nDollar Performance:")
print(f"  Initial capital: ${INITIAL_CAPITAL:,.0f}")
print(f"  Final capital: ${final_capital:,.0f}")
print(f"  Total P&L: ${total_pnl:,.0f}")
print(f"  Total return: {(final_capital/INITIAL_CAPITAL - 1)*100:.2f}%")

# Best and worst years
best_year = portfolio_df.loc[portfolio_df['portfolio_return'].idxmax()]
worst_year = portfolio_df.loc[portfolio_df['portfolio_return'].idxmin()]

print(f"\nBest Year:")
print(f"  {best_year['formation_year']:.0f}→{best_year['holding_year']:.0f}: "
      f"{best_year['portfolio_return']*100:.2f}% (P&L: ${best_year['total_dollar_pnl']:,.0f})")

print(f"\nWorst Year:")
print(f"  {worst_year['formation_year']:.0f}→{worst_year['holding_year']:.0f}: "
      f"{worst_year['portfolio_return']*100:.2f}% (P&L: ${worst_year['total_dollar_pnl']:,.0f})")

# ============================================================================
# DETAILED ANNUAL RESULTS
# ============================================================================
print("\n" + "="*80)
print("ANNUAL PORTFOLIO RETURNS")
print("="*80)

print(f"\n{'Formation':<10} {'Holding':<10} {'Long':<12} {'Short':<12} {'Portfolio':<12} {'Spread':<12} {'P&L ($)':<15} {'Capital ($)':<15}")
print("-"*120)
for _, row in portfolio_df.iterrows():
    print(f"{row['formation_year']:<10.0f} {row['holding_year']:<10.0f} "
          f"{row['long_return']*100:>10.2f}% {row['short_return']*100:>10.2f}% "
          f"{row['portfolio_return']*100:>10.2f}% {row['spread']*100:>10.2f}% "
          f"{row['total_dollar_pnl']:>13,.0f} {row['total_capital']:>13,.0f}")

# ============================================================================
# SAVE RESULTS
# ============================================================================
print("\n[STEP 4] Saving results...")

# Save portfolio performance
portfolio_path = output_dir / 'long_short_portfolio_results.csv'
portfolio_df.to_csv(portfolio_path, index=False)
print(f"✓ Portfolio results: {portfolio_path}")

# Create summary statistics
summary_stats = {
    'Metric': [
        'Total Years',
        'Average Annual Return (%)',
        'Volatility (%)',
        'Sharpe Ratio',
        'Win Rate (%)',
        'Total Return (%)',
        'Initial Capital ($)',
        'Final Capital ($)',
        'Total P&L ($)',
        'Average Long Return (%)',
        'Average Short Return (%)',
        'Average Spread (%)'
    ],
    'Value': [
        total_years,
        f"{avg_portfolio_return:.2f}",
        f"{volatility_portfolio:.2f}",
        f"{sharpe_portfolio:.3f}",
        f"{win_rate_portfolio:.1f}",
        f"{total_return_portfolio:.2f}",
        f"{INITIAL_CAPITAL:,.0f}",
        f"{final_capital:,.0f}",
        f"{total_pnl:,.0f}",
        f"{avg_long_return:.2f}",
        f"{avg_short_return:.2f}",
        f"{avg_spread:.2f}"
    ]
}
summary_df = pd.DataFrame(summary_stats)
summary_path = output_dir / 'portfolio_summary_statistics.csv'
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary statistics: {summary_path}")


📊 Creating long/short portfolios...
💰 Total capital: $1,000,000
   Capital allocation: Flexible based on number of positions

📈 Building portfolios - VERSION 1: Fixed Top 100 / Bottom 100
------------------------------------------------------------

[STEP 2] Building long-short portfolio strategy...

Backtesting from 2003 to 2024
Strategy: Sort by predicted return in Year T, hold in Year T+1
  Skipping 2003: Not enough stocks
  Skipping 2004: Not enough stocks
      ticker  form_year     alpha   beta_mkt    beta_smb   beta_hml  \
49      CECO     2006.0  0.361373  19.125955  -24.577236  -0.117281   
507     SNTO     2006.0  2.948731  73.102670 -106.483063 -45.330988   
829       AP     2006.0  0.466875   9.787052  -18.835112   1.954331   
848      AXR     2006.0  0.779322  33.814595  -45.151610 -12.229296   
914     ANDE     2006.0  0.437526  32.509664  -35.621660  -9.383441   
...      ...        ...       ...        ...         ...        ...   
41575   ADLR     2006.0  0.435750  53